# Лабораторная работа: Кластеризация текстовых данных о психическом здоровье
## Цель: Применить методы кластеризации к текстам и сравнить с реальными метками

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Для визуализации
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = [12, 8]

# Попробуем импортировать UMAP
try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print("UMAP не установлен. Установите: pip install umap-learn")

class TextClusterizer:
    """
    Класс для кластеризации текстовых данных.
    """
    
    def __init__(self, random_state=42):
        self.random_state = random_state
        self.vectorizer = None
        self.X_tfidf = None
        self.labels_true = None
        self.label_names = None
        
    def load_and_prepare(self, filepath):
        """Загрузка и подготовка данных"""
        print("=" * 60)
        print("ЗАГРУЗКА ДАННЫХ")
        print("=" * 60)
        
        # Загружаем данные
        df = pd.read_csv(filepath)
        print(f"Загружено {len(df)} записей")
        print(f"Распределение классов:\n{df['status'].value_counts()}")
        
        # Кодируем метки
        self.label_names = df['status'].unique()
        le = LabelEncoder()
        self.labels_true = le.fit_transform(df['status'])
        
        # Векторизуем тексты
        print("\nВекторизация текстов (TF-IDF)...")
        self.vectorizer = TfidfVectorizer(
            max_features=5000,  # Ограничим количество признаков
            stop_words='english',
            min_df=2,
            max_df=0.95,
            ngram_range=(1, 2)
        )
        self.X_tfidf = self.vectorizer.fit_transform(df['text'].fillna(''))
        print(f"Размер матрицы признаков: {self.X_tfidf.shape}")
        
        return df
    
    def plot_cluster_comparison(self, X_reduced, labels_pred, title):
        """Визуализация результатов кластеризации с реальными метками"""
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        
        # Реальные метки
        scatter1 = axes[0].scatter(X_reduced[:, 0], X_reduced[:, 1], 
                                   c=self.labels_true, cmap='viridis', 
                                   alpha=0.7, s=30)
        axes[0].set_title(f'Реальные классы\n{self.label_names}')
        axes[0].set_xlabel('Component 1')
        axes[0].set_ylabel('Component 2')
        
        # Предсказанные кластеры
        scatter2 = axes[1].scatter(X_reduced[:, 0], X_reduced[:, 1], 
                                   c=labels_pred, cmap='tab10', 
                                   alpha=0.7, s=30)
        axes[1].set_title(f'Кластеры ({title})')
        axes[1].set_xlabel('Component 1')
        axes[1].set_ylabel('Component 2')
        
        plt.tight_layout()
        plt.show()
        
    def apply_kmeans(self, n_clusters=None):
        """Применение K-Means кластеризации"""
        print("\n" + "=" * 60)
        print("K-MEANS КЛАСТЕРИЗАЦИЯ")
        print("=" * 60)
        
        # Определяем оптимальное количество кластеров
        if n_clusters is None:
            inertias = []
            K_range = range(2, 8)
            for k in K_range:
                kmeans = KMeans(n_clusters=k, random_state=self.random_state, n_init=10)
                kmeans.fit(self.X_tfidf)
                inertias.append(kmeans.inertia_)
            
            # График elbow
            plt.figure(figsize=(10, 4))
            plt.subplot(1, 2, 1)
            plt.plot(K_range, inertias, 'bo-')
            plt.xlabel('k')
            plt.ylabel('Inertia')
            plt.title('Метод локтя для K-Means')
            plt.grid(True, alpha=0.3)
            
            # Силуэт
            from sklearn.metrics import silhouette_score
            sil_scores = []
            for k in K_range:
                kmeans = KMeans(n_clusters=k, random_state=self.random_state, n_init=10)
                labels = kmeans.fit_predict(self.X_tfidf)
                if len(set(labels)) > 1:
                    sil_scores.append(silhouette_score(self.X_tfidf, labels))
                else:
                    sil_scores.append(0)
            
            plt.subplot(1, 2, 2)
            plt.plot(K_range, sil_scores, 'ro-')
            plt.xlabel('k')
            plt.ylabel('Silhouette Score')
            plt.title('Силуэтный коэффициент')
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()
            
            n_clusters = K_range[np.argmax(sil_scores)]
            print(f"Выбрано оптимальное k = {n_clusters}")
        
        # Применяем K-Means
        kmeans = KMeans(n_clusters=n_clusters, random_state=self.random_state, n_init=10)
        labels_kmeans = kmeans.fit_predict(self.X_tfidf)
        
        # Оценка качества
        print(f"\nРезультаты K-Means (k={n_clusters}):")
        print(f"  Silhouette Score: {silhouette_score(self.X_tfidf, labels_kmeans):.4f}")
        print(f"  Adjusted Rand Index: {adjusted_rand_score(self.labels_true, labels_kmeans):.4f}")
        print(f"  Normalized Mutual Info: {normalized_mutual_info_score(self.labels_true, labels_kmeans):.4f}")
        
        # Визуализация через PCA
        pca = PCA(n_components=2, random_state=self.random_state)
        X_pca = pca.fit_transform(self.X_tfidf.toarray())
        self.plot_cluster_comparison(X_pca, labels_kmeans, f'K-Means (k={n_clusters})')
        
        return labels_kmeans
    
    def apply_dbscan(self, eps=0.5, min_samples=5):
        """Применение DBSCAN кластеризации"""
        print("\n" + "=" * 60)
        print("DBSCAN КЛАСТЕРИЗАЦИЯ")
        print("=" * 60)
        
        # Пробуем разные параметры
        eps_values = [0.3, 0.5, 0.7, 1.0]
        
        fig, axes = plt.subplots(2, 2, figsize=(14, 12))
        axes = axes.flatten()
        
        best_score = -1
        best_labels = None
        best_params = None
        
        # Сначала уменьшаем размерность для DBSCAN
        svd = TruncatedSVD(n_components=50, random_state=self.random_state)
        X_reduced = svd.fit_transform(self.X_tfidf)
        
        for idx, eps_val in enumerate(eps_values):
            dbscan = DBSCAN(eps=eps_val, min_samples=min_samples)
            labels = dbscan.fit_predict(X_reduced)
            
            n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
            n_noise = list(labels).count(-1)
            
            # Визуализация через PCA
            pca = PCA(n_components=2, random_state=self.random_state)
            X_pca = pca.fit_transform(X_reduced)
            
            scatter = axes[idx].scatter(X_pca[:, 0], X_pca[:, 1], 
                                        c=labels, cmap='tab10', alpha=0.7, s=30)
            axes[idx].set_title(f'DBSCAN (eps={eps_val}, min_samples={min_samples})\n'
                               f'Кластеров: {n_clusters}, Шума: {n_noise}')
            axes[idx].set_xlabel('PC1')
            axes[idx].set_ylabel('PC2')
            
            # Оценка
            if n_clusters > 1:
                # Силуэт только для не-шумовых точек
                mask = labels != -1
                if mask.sum() > 0 and len(set(labels[mask])) > 1:
                    score = silhouette_score(X_reduced[mask], labels[mask])
                    print(f"  eps={eps_val}: {n_clusters} кластеров, шум={n_noise}, silhouette={score:.4f}")
                    if score > best_score:
                        best_score = score
                        best_labels = labels
                        best_params = (eps_val, min_samples)
            else:
                print(f"  eps={eps_val}: {n_clusters} кластеров, шум={n_noise}")
        
        plt.tight_layout()
        plt.show()
        
        if best_labels is not None:
            print(f"\nЛучшие параметры: eps={best_params[0]}, min_samples={best_params[1]}")
            print(f"ARI с реальными метками: {adjusted_rand_score(self.labels_true, best_labels):.4f}")
        
        return best_labels
    
    def apply_agglomerative(self, n_clusters=None):
        """Иерархическая кластеризация"""
        print("\n" + "=" * 60)
        print("ИЕРАРХИЧЕСКАЯ КЛАСТЕРИЗАЦИЯ")
        print("=" * 60)
        
        # Сначала уменьшаем размерность для ускорения
        svd = TruncatedSVD(n_components=100, random_state=self.random_state)
        X_reduced = svd.fit_transform(self.X_tfidf)
        
        if n_clusters is None:
            n_clusters = len(np.unique(self.labels_true))
        
        # Пробуем разные методы linkage
        linkages = ['ward', 'complete', 'average']
        
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        
        best_score = -1
        best_labels = None
        
        for idx, linkage in enumerate(linkages):
            agg = AgglomerativeClustering(n_clusters=n_clusters, linkage=linkage)
            labels = agg.fit_predict(X_reduced)
            
            pca = PCA(n_components=2, random_state=self.random_state)
            X_pca = pca.fit_transform(X_reduced)
            
            axes[idx].scatter(X_pca[:, 0], X_pca[:, 1], 
                             c=labels, cmap='tab10', alpha=0.7, s=30)
            axes[idx].set_title(f'Agglomerative ({linkage})')
            axes[idx].set_xlabel('PC1')
            axes[idx].set_ylabel('PC2')
            
            score = silhouette_score(X_reduced, labels)
            ari = adjusted_rand_score(self.labels_true, labels)
            print(f"  linkage={linkage}: silhouette={score:.4f}, ARI={ari:.4f}")
            
            if score > best_score:
                best_score = score
                best_labels = labels
        
        plt.tight_layout()
        plt.show()
        
        return best_labels
    
    def visualize_clusters_2d_3d(self, labels_pred, method_name):
        """2D и 3D визуализация кластеров"""
        print(f"\nВизуализация {method_name} в 2D и 3D...")
        
        # PCA для 3D
        pca = PCA(n_components=3, random_state=self.random_state)
        X_pca = pca.fit_transform(self.X_tfidf.toarray())
        
        fig = plt.figure(figsize=(15, 6))
        
        # 2D
        ax1 = fig.add_subplot(121)
        scatter = ax1.scatter(X_pca[:, 0], X_pca[:, 1], 
                              c=labels_pred, cmap='tab10', alpha=0.7, s=20)
        ax1.set_xlabel('PC1')
        ax1.set_ylabel('PC2')
        ax1.set_title(f'{method_name} - 2D проекция')
        
        # 3D
        ax2 = fig.add_subplot(122, projection='3d')
        scatter3d = ax2.scatter3D(X_pca[:, 0], X_pca[:, 1], X_pca[:, 2], 
                                  c=labels_pred, cmap='tab10', alpha=0.7, s=20)
        ax2.set_xlabel('PC1')
        ax2.set_ylabel('PC2')
        ax2.set_zlabel('PC3')
        ax2.set_title(f'{method_name} - 3D проекция')
        
        plt.tight_layout()
        plt.show()
    
    def plot_confusion_matrix(self, labels_pred, method_name):
        """Матрица соответствия между кластерами и реальными классами"""
        from sklearn.metrics import confusion_matrix
        
        # Строим матрицу соответствия
        cm = confusion_matrix(self.labels_true, labels_pred)
        
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=range(cm.shape[1]),
                    yticklabels=self.label_names)
        plt.xlabel('Кластеры')
        plt.ylabel('Реальные классы')
        plt.title(f'Матрица соответствия: {method_name}')
        plt.tight_layout()
        plt.show()
    
    def find_top_words_per_cluster(self, labels_pred, n_words=10):
        """Поиск ключевых слов для каждого кластера"""
        print("\n" + "=" * 60)
        print("КЛЮЧЕВЫЕ СЛОВА ПО КЛАСТЕРАМ")
        print("=" * 60)
        
        feature_names = self.vectorizer.get_feature_names_out()
        unique_clusters = np.unique(labels_pred)
        
        for cluster in unique_clusters:
            if cluster == -1:  # Шум в DBSCAN
                continue
                
            # Находим документы в кластере
            cluster_docs = self.X_tfidf[labels_pred == cluster]
            
            if cluster_docs.shape[0] == 0:
                continue
                
            # Суммируем TF-IDF по всем документам кластера
            cluster_center = cluster_docs.sum(axis=0).A1
            
            # Находим топ слов
            top_indices = cluster_center.argsort()[-n_words:][::-1]
            top_words = [feature_names[i] for i in top_indices]
            
            print(f"\nКластер {cluster} ({cluster_docs.shape[0]} документов):")
            print(f"  Ключевые слова: {', '.join(top_words)}")
    
    def run_full_analysis(self, filepath):
        """Запуск полного анализа кластеризации"""
        # Загрузка данных
        df = self.load_and_prepare(filepath)
        
        # Применяем разные методы кластеризации
        labels_kmeans = self.apply_kmeans(n_clusters=len(self.label_names))
        self.visualize_clusters_2d_3d(labels_kmeans, "K-Means")
        self.plot_confusion_matrix(labels_kmeans, "K-Means")
        self.find_top_words_per_cluster(labels_kmeans)
        
        labels_agg = self.apply_agglomerative(n_clusters=len(self.label_names))
        self.plot_confusion_matrix(labels_agg, "Agglomerative")
        
        # DBSCAN
        self.apply_dbscan(eps=0.7, min_samples=3)
        
        # Сравнение методов
        self.compare_clustering_methods()
        
        print("\n" + "=" * 60)
        print("АНАЛИЗ ЗАВЕРШЕН")
        print("=" * 60)
        
    def compare_clustering_methods(self):
        """Сравнение всех методов кластеризации"""
        print("\n" + "=" * 60)
        print("СРАВНЕНИЕ МЕТОДОВ КЛАСТЕРИЗАЦИИ")
        print("=" * 60)
        
        methods = {
            'K-Means': lambda: KMeans(n_clusters=len(self.label_names), 
                                      random_state=self.random_state, n_init=10),
            'Agglomerative': lambda: AgglomerativeClustering(n_clusters=len(self.label_names))
        }
        
        # Сначала уменьшаем размерность для DBSCAN
        svd = TruncatedSVD(n_components=50, random_state=self.random_state)
        X_reduced = svd.fit_transform(self.X_tfidf)
        
        results = []
        
        for name, method_func in methods.items():
            try:
                model = method_func()
                if name == 'K-Means':
                    labels = model.fit_predict(self.X_tfidf)
                else:
                    labels = model.fit_predict(self.X_tfidf.toarray())
                
                ari = adjusted_rand_score(self.labels_true, labels)
                nmi = normalized_mutual_info_score(self.labels_true, labels)
                
                if len(set(labels)) > 1:
                    sil = silhouette_score(self.X_tfidf, labels)
                else:
                    sil = 0
                    
                results.append({
                    'Метод': name,
                    'ARI': ari,
                    'NMI': nmi,
                    'Silhouette': sil
                })
            except Exception as e:
                print(f"Ошибка при {name}: {e}")
        
        # DBSCAN
        dbscan = DBSCAN(eps=0.7, min_samples=3)
        labels_db = dbscan.fit_predict(X_reduced)
        if len(set(labels_db)) > 1:
            mask = labels_db != -1
            if mask.sum() > 0:
                sil_db = silhouette_score(X_reduced[mask], labels_db[mask])
            else:
                sil_db = 0
        else:
            sil_db = 0
        results.append({
            'Метод': 'DBSCAN',
            'ARI': adjusted_rand_score(self.labels_true, labels_db),
            'NMI': normalized_mutual_info_score(self.labels_true, labels_db),
            'Silhouette': sil_db
        })
        
        # Вывод результатов
        results_df = pd.DataFrame(results)
        print(results_df.to_string(index=False))
        
        # Визуализация сравнения
        fig, ax = plt.subplots(figsize=(10, 6))
        results_df.plot(x='Метод', y=['ARI', 'NMI'], kind='bar', ax=ax)
        ax.set_title('Сравнение качества кластеризации')
        ax.set_ylabel('Score')
        ax.set_ylim(0, 1)
        ax.legend(loc='lower right')
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()


# Запуск анализа
if __name__ == "__main__":
    # Создаем экземпляр класса
    clusterizer = TextClusterizer(random_state=42)
    
    # Запускаем полный анализ
    clusterizer.run_full_analysis('mental_health_combined_test.csv')

ЗАГРУЗКА ДАННЫХ
Загружено 992 записей
Распределение классов:
status
Anxiety       248
Depression    248
Normal        248
Suicidal      248
Name: count, dtype: int64

Векторизация текстов (TF-IDF)...
Размер матрицы признаков: (992, 5000)

K-MEANS КЛАСТЕРИЗАЦИЯ


  File "D:\Anaconda\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "D:\Anaconda\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "D:\Anaconda\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                        pass_fds, cwd, env,
                        ^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
                        gid, gids, uid, umask,
                        ^^^^^^^^^^^^^^^^^^^^^^
                        start_new_session, process_group)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "D:\Anaconda\Lib\subprocess.py", line 1554, in _execute_child
    hp, ht, pid, t


Результаты K-Means (k=4):


UnboundLocalError: cannot access local variable 'silhouette_score' where it is not associated with a value